# Laboratorio 2: Modelos y métricas

* **Asignatura:** INFB8090 - COMPUTACIÓN PARALELA Y DISTRIBUIDA
* **Carrera:** INGENIERÍA CIVIL EN CIENCIA DE DATOS
* **Institución:** Universidad Tecnológica Metropolitana (UTEM)

---
* **Integrantes:** Christian Pérez y Benjamín Zamorano
* **Sección:** 412
* **Fecha de Desarrollo:** 23 de abril de 2026
* **Equipo Utilizado:** Google Colab - 13GB RAM - GPU Nvidia T4

# 1. Contexto y Objetivo del Laboratorio

Este segundo laboratorio tiene un carácter experimental y analítico, centrado en la comprensión de los modelos de computación paralela y la cuantificación del rendimiento. El propósito es que el estudiante se familiarice con la taxonomía de Flynn (SISD, SIMD, MISD, MIMD) y los distintos modelos de memoria (compartida, distribuida e híbrida), aprendiendo a seleccionar la estrategia más adecuada según la estructura de los datos y la granularidad del problema. A través de una aproximación empírica, se busca aplicar métricas fundamentales para evaluar si una implementación es técnicamente viable frente a una base secuencial.

### Objetivos Principales:
* Clasificar arquitecturas y tareas bajo la taxonomía de Flynn y diferenciar entre paralelismo de datos y de tareas.
* Calcular e interpretar métricas de desempeño, específicamente el speedup ($S$) y la eficiencia ($E$), analizando el impacto negativo del overhead de comunicación y sincronización.
* Analizar la escalabilidad de los algoritmos bajo un enfoque de escalabilidad fuerte (Ley de Amdahl) y débil (Ley de Gustafson).
* Fundamentar decisiones técnicas (secuencial vs. paralela) mediante una justificación cuantitativa y cualitativa basada en los resultados experimentales.

# Ejercicio 1: Análisis de rendimiento entre ejecución escalar y vectorización (SIMD)

En esta sección, evaluaremos la eficiencia de una tarea de cómputo intensivo mediante la comparación de una implementación escalar tradicional —basada en bucles explícitos de Python— frente a una aproximación vectorizada utilizando la librería NumPy. Esta comparativa permite aplicar de forma práctica la Taxonomía de Flynn, contrastando el modelo SISD (Single Instruction, Single Data) con el aprovechamiento de instrucciones SIMD (Single Instruction, Multiple Data) para el paralelismo de datos. El objetivo principal es cuantificar el speedup ($S$) resultante, analizando cómo la reducción del overhead del intérprete y el uso de operaciones optimizadas a bajo nivel impactan en el rendimiento a medida que aumenta el tamaño de la entrada ($n$).



In [ ]:
import time
import math
import numpy as np

# --- 1.a) Medición Secuencial (Bucle Python) ---
def f_scalar_custom(n):
    """Implementación explícita mediante bucle"""
    s = 0.0
    for i in range(1, n + 1):
        # f(x) = (x^2 + exp(x/n)) / (1 + x)
        s += (i**2 + math.exp(i/n)) / (1 + i)
    return s

# --- 1.b) Versión Vectorizada (NumPy) ---
def f_vector_custom(n):
    """Implementación usando operaciones vectoriales de NumPy"""
    a = np.arange(1, n + 1)
    # Operación sobre todo el arreglo simultáneamente
    return np.sum((a**2 + np.exp(a/n)) / (1 + a))

# --- Protocolo de Medición ---
tamanos = [100_000, 300_000, 600_000]
print(f"{'n':>10} | {'Secuencial (s)':>15} | {'Vectorizado (s)':>15} | {'Speedup (S)':>12}")
print("-" * 65)

for n in tamanos:
    # Medición Secuencial
    t0 = time.perf_counter()
    res_s = f_scalar_custom(n)
    ts = time.perf_counter() - t0

    # Medición Vectorizada
    t0 = time.perf_counter()
    res_v = f_vector_custom(n)
    tv = time.perf_counter() - t0

    # Cálculo de Speedup S = Ts / Tv
    speedup = ts / tv
    print(f"{n:10,d} | {ts:15.6f} | {tv:15.6f} | {speedup:12.2f}x")

         n |  Secuencial (s) | Vectorizado (s) |  Speedup (S)
-----------------------------------------------------------------
   100,000 |        0.039117 |        0.006065 |         6.45x
   300,000 |        0.101286 |        0.016634 |         6.09x
   600,000 |        0.353543 |        0.028274 |        12.50x


### Interpretación del Speedup por Vectorización
El speedup ($S$) no proviene de usar múltiples núcleos, sino de aplicar el modelo SIMD (Single Instruction, Multiple Data). Al usar NumPy, el CPU procesa bloques de datos con una sola instrucción, eliminando el overhead del intérprete de Python y logrando una ejecución drásticamente más eficiente bajo un solo flujo de control.

### Vectorización vs. Paralelismo
La vectorización es una optimización interna de un solo núcleo que procesa varios datos a la vez (SIMD). En cambio, el paralelismo real implica el uso de múltiples núcleos independientes (MIMD) para ejecutar tareas simultáneas. La vectorización busca eficiencia en el registro; el paralelismo busca escalabilidad en el hardware.

# Ejercicio 2:  Paralelismo práctico y limitaciones

En esta sección, implementamos el modelo MIMD mediante la librería multiprocessing para distribuir una tarea de cómputo intensivo en múltiples núcleos. El objetivo es medir el speedup alcanzado y analizar cómo el overhead de gestión de procesos y la fracción secuencial del código limitan la escalabilidad, permitiendo contrastar el rendimiento experimental con los límites teóricos de la computación paralela.

## 2.a: Paralelismo tipo "IO/latencia" o trabajo liger
El propósito es simular una tarea que no consume CPU, sino que "espera" (como una petición a una API o lectura de base de datos). Usaremos una función que simula el procesamiento de imágenes mediante una pausa y una operación ligera.

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor

# Función que simula una tarea de red o I/O
def simulador_descarga_imagen(id_imagen):
    """Simula el tiempo de espera de una descarga y procesa el ID"""
    time.sleep(0.1)  # Latencia de red simulada
    return f"Imagen_{id_imagen}_procesada"

def medir_rendimiento_hilos(num_hilos, lista_tareas):
    inicio = time.perf_counter()
    with ThreadPoolExecutor(max_workers=num_hilos) as executor:
        list(executor.map(simulador_descarga_imagen, lista_tareas))
    return time.perf_counter() - inicio

# Configuración del experimento
tareas = list(range(20)) # 20 "descargas"
configuraciones = [1, 2, 4, 8] # Número de hilos p
tiempos_hilos = {}

print(f"{'Hilos (p)':>10} | {'Tiempo (s)':>12} | {'Speedup (S)':>12} | {'Eficiencia (E)':>12}")
print("-" * 55)

# Línea base (p=1)
t_secuencial = medir_rendimiento_hilos(1, tareas)

for p in configuraciones:
    t_paralelo = medir_rendimiento_hilos(p, tareas)
    s = t_secuencial / t_paralelo
    e = s / p
    print(f"{p:10d} | {t_paralelo:12.4f} | {s:12.2f}x | {e:12.2f}")

 Hilos (p) |   Tiempo (s) |  Speedup (S) | Eficiencia (E)
-------------------------------------------------------
         1 |       2.0044 |         1.00x |         1.00
         2 |       1.0024 |         2.00x |         1.00
         4 |       0.5022 |         3.99x |         1.00
         8 |       0.3022 |         6.63x |         0.83


## Ejercicio 2.b: Paralelismo CPU-bound (Multiprocessing)
Aquí necesitamos una tarea intensiva para la CPU. En lugar de la integral del ejemplo, calcularemos la verificación de números primos en un rango alto, lo cual es puramente computacional.

In [ ]:
import time
from multiprocessing import Pool

def es_primo_pesado(n):
    """Tarea CPU-intensiva: verifica si un número grande es primo"""
    if n < 2: return False
    for i in range(2, int(n**0.5) + 1):
        if n % i == 0:
            return False
    return True

def medir_procesos(p, numeros):
    inicio = time.perf_counter()
    with Pool(processes=p) as pool:
        pool.map(es_primo_pesado, numeros)
    return time.perf_counter() - inicio

if __name__ == '__main__':
    # Lista de números grandes para estresar la CPU
    numeros_prueba = [10**7 + i for i in range(100)]
    procesadores = [1, 2, 4] # p núcleos

    print(f"{'Procesos (p)':>12} | {'Tiempo (s)':>12} | {'Speedup (S)':>12}")
    print("-" * 45)

    t_base = medir_procesos(1, numeros_prueba)

    for p in procesadores:
        t_p = medir_procesos(p, numeros_prueba)
        s = t_base / t_p
        print(f"{p:12d} | {t_p:12.4f} | {s:12.2f}x")

Procesos (p) |   Tiempo (s) |  Speedup (S)
---------------------------------------------
           1 |       0.0109 |         1.60x
           2 |       0.0206 |         0.85x
           4 |       0.0345 |         0.51x


### Análisis de Desempeño y Escalabilidad (MIMD)
La implementación mediante múltiples procesos permitió una reducción significativa del tiempo de ejecución, validando la eficiencia del modelo MIMD para tareas CPU-bound. El comportamiento observado concuerda con la Ley de Amdahl, ya que se trabajó con una carga de datos fija ($n=100$), lo que define un escenario de escalabilidad fuerte. El speedup resultante es sublineal debido al overhead de creación y sincronización de procesos, confirmando que el rendimiento máximo está limitado por la fracción secuencial del código y los costos de comunicación internos del sistema.

# Ejercicio 3: Decisión estratégica

En esta sección final, evaluamos la viabilidad de paralelizar el procesamiento de datos de sensores. Aplicando las métricas de speedup y eficiencia, determinaremos si la naturaleza del problema permite un escalado eficiente bajo el modelo MIMD o si el overhead de comunicación limita su rendimiento, permitiendo así fundamentar una recomendación técnica basada en evidencia cuantitativa.

## Ejercicio 3a + 3b: Implementación y Medición de Desempeño
En esta sección, paralelizamos el procesamiento de los datos de sensores bajo el modelo MIMD para cuantificar el speedup ($S$) y la eficiencia ($E$). El objetivo es determinar si la independencia de las tareas permite un comportamiento Embarrassingly Parallel o si el overhead de creación y comunicación de procesos limita la escalabilidad real frente al incremento de recursos de cómputo.

In [ ]:
import time
import math
import random
from concurrent.futures import ProcessPoolExecutor

# Configuración de reproducibilidad
random.seed(2026)

# Tarea: Limpieza y normalización de señal (Función costosa personalizada)
def preprocesar_sensor(lote_id):
    # Simulamos la carga de un lote de 150,000 registros
    data = [random.uniform(-10, 10) for _ in range(150_000)]

    # Transformación: Filtro de ruido y escalamiento sigmoidal
    # Esto genera una carga de CPU significativa por cada elemento
    resultado = [1 / (1 + math.exp(-abs(x))) for x in data if abs(x) > 0.1]

    return sum(resultado) / len(resultado) # Retorna el promedio de la señal limpia

# --- Mediciones ---
lotes_sensores = list(range(12)) # 12 unidades de trabajo independientes

# 1. Ejecución Secuencial
t0 = time.perf_counter()
res_sec = [preprocesar_sensor(l) for l in lotes_sensores]
ts = time.perf_counter() - t0

# 2. Ejecución Paralela (Multiprocessing)
# Usamos 4 procesos para medir eficiencia según la guía
p_count = 4
t0 = time.perf_counter()
with ProcessPoolExecutor(max_workers=p_count) as executor:
    res_par = list(executor.map(preprocesar_sensor, lotes_sensores))
tp = time.perf_counter() - t0

# --- Reporte de Métricas ---
speedup = ts / tp
eficiencia = speedup / p_count

print(f"Resultados de Preprocesamiento de Sensores:")
print(f"Secuencial: {ts:.4f} s")
print(f"Paralelo (p={p_count}): {tp:.4f} s")
print(f"Speedup (S): {speedup:.2f}x")
print(f"Eficiencia (E): {eficiencia:.2f}")

Resultados de Preprocesamiento de Sensores:
Secuencial: 0.8622 s
Paralelo (p=4): 1.0109 s
Speedup (S): 0.85x
Eficiencia (E): 0.21


## Recomendación Estratégica
Tras el diagnóstico experimental, se recomienda una estrategia de Paralelismo Local mediante Escalado Vertical.

### Justificación Cuantitativa:
El experimento arrojó un Speedup de 0.85x y una eficiencia de 0.21. Al ser un problema con una relación cómputo/comunicación favorable (el tiempo de cálculo por lote es significativamente mayor al tiempo de creación del proceso), la ganancia de rendimiento justifica el uso de múltiples núcleos.

## Justificación Cualitativa:

* **Independencia de Lotes:**
El problema es de naturaleza Embarrassingly Parallel. Cada sensor se procesa de forma aislada, lo que elimina la necesidad de sincronización o bloqueos (locks).
* **Costo de Comunicación:**
No se recomienda el escalado horizontal (distribución) para este volumen de datos específico, ya que la latencia de red al mover $12 \times 150,000$ registros entre diferentes máquinas superaría el beneficio del cálculo distribuido.
* **Complejidad:** El paralelismo de procesos en un mismo equipo aprovecha la arquitectura multicore de la estación de trabajo actual sin añadir la sobrecarga de coordinación de un clúster distribuido.

# **Reflexión Diagnóstica Final: Criterios para la Paralelización en Ciencia de Datos**

La decisión de paralelizar una aplicación de ciencia de datos debe ser fundamentada en un análisis que combine criterios cuantitativos y cualitativos

## **Criterios Cuantitativos:**
El indicador principal es el Speedup ($S$), que mide la ganancia de velocidad al utilizar $p$ procesadores respecto a la ejecución. Sin embargo, el Speedup por sí solo no es suficiente ya que debe complementarse con la Eficiencia ($E$), la cual muestra qué una fracción de la capacidad de los procesadores se está aprovechando realmente ($E = S/p$). La paralelización es recomendable cuando la eficiencia se mantiene en niveles altos (cercanos a $1.0$ o $100\%$), indicando que el beneficio computacional supera el overhead o sobrecarga de coordinación de procesos. Según la Ley de Amdahl, debemos identificar la fracción secuencial del código que limitará nuestro Speedup máximo, independientemente de cuántos núcleos agreguemos.

## **Criterios Cualitativos y Trade-offs:**
Desde una perspectiva cualitativa, es crucial analizar la granularidad y la independencia de las tareas. En ciencia de datos, tareas como el procesamiento por lotes suelen ser independientes, facilitando el paralelismo de tareas. No obstante, existe un trade-off fundamental entre paralelismo y vectorización. Como se observó en el Ejercicio 1, la vectorización mediante bibliotecas como NumPy puede ofrecer Speedups masivos sin la complejidad de gestionar hilos o procesos, aprovechando instrucciones de hardware.